# Dense Passage Retrieval (DPR) Evaluation Pipeline — Local Indexes & CLI Commands

Uses **already-downloaded local Pyserini indexes** (`C:\Users\Dwija Patel\.cache\pyserini\indexes\`) without re-downloading, and executes `sys.executable` (`{py}`) CLI commands (`pyserini.search.lucene`, `pyserini.eval.convert_trec_run_to_dpr_retrieval_run`, `pyserini.eval.evaluate_dpr_retrieval`) across:
- **All 5 datasets (`nq`, `trivia`, `wq`, `curated`, `squad`)** on the **Multi-dataset index** (`faiss.wikipedia-dpr-100w.dpr_multi.*`) and **BM25** (`lucene-inverted.wikipedia-dpr-100w.*`).
- **NQ only (`nq`)** on the **NQ-only index** (`faiss.wikipedia-dpr-100w.dpr_single-nq.*`).

## 1. Imports & Local Index Auto-Discovery 

In [1]:
import os
import re
import sys
import json
import struct
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import DPRQuestionEncoder, DPRQuestionEncoderTokenizer
from pyserini.search import get_topics

py = sys.executable
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

RUNS_DIR = Path('runs')
CKPT_DIR = Path('checkpoints')
RUNS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Resolve your existing local Pyserini index cache directly
INDEX_CACHE_DIR = Path(os.path.expanduser('~')) / '.cache' / 'pyserini' / 'indexes'

def find_local_index(prefix: str, cache_dir: Path = INDEX_CACHE_DIR) -> str:
    matches = sorted([p for p in cache_dir.glob(f'{prefix}*') if p.is_dir()])
    if not matches:
        raise FileNotFoundError(f'Could not find local index matching {prefix}* inside {cache_dir}')
    return str(matches[-1])

root = find_local_index('lucene-inverted.wikipedia-dpr-100w')
faiss_multi_root = find_local_index('faiss.wikipedia-dpr-100w.dpr_multi')
faiss_single_nq_root = find_local_index('faiss.wikipedia-dpr-100w.dpr_single-nq')

print('Python executable (py):', py)
print('Device:', DEVICE)
print('Lucene root:', root)
print('FAISS Multi root:', faiss_multi_root)
print('FAISS Single-NQ root:', faiss_single_nq_root)

Python executable (py): c:\Users\Dwija Patel\dwija\sem5\indexing\winenv\Scripts\python.exe
Device: cpu
Lucene root: C:\Users\Dwija Patel\.cache\pyserini\indexes\lucene-inverted.wikipedia-dpr-100w.20260508.deb4c7b
FAISS Multi root: C:\Users\Dwija Patel\.cache\pyserini\indexes\faiss.wikipedia-dpr-100w.dpr_multi.20200127.f403c3
FAISS Single-NQ root: C:\Users\Dwija Patel\.cache\pyserini\indexes\faiss.wikipedia-dpr-100w.dpr_single-nq.20200115.cd5034


## 2. Dataset & Index Routing Configuration

- **Index built on all datasets (`faiss_multi_root`)** $\rightarrow$ evaluated on **all 5 datasets** (`nq`, `trivia`, `wq`, `curated`, `squad`).
- **Index built only on NQ (`faiss_single_nq_root`)** $\rightarrow$ evaluated **strictly on NQ** (`nq`).

In [2]:
ALL_DATASETS = {
    'nq':      {'topics': 'dpr-nq-test',      'tag': 'nq-test',      'regex': False},
    'trivia':  {'topics': 'dpr-trivia-test',  'tag': 'trivia-test',  'regex': False},
    'wq':      {'topics': 'dpr-wq-test',      'tag': 'wq-test',      'regex': False},
    'curated': {'topics': 'dpr-curated-test', 'tag': 'curated-test', 'regex': True},
    'squad':   {'topics': 'dpr-squad-test',   'tag': 'squad-test',   'regex': False},
}

NQ_ONLY_DATASETS = {
    'nq': ALL_DATASETS['nq']
}

DENSE_CONFIGS = {
    'multi': {
        'local_index_dir': faiss_multi_root,
        'encoder_name': 'facebook/dpr-question_encoder-multiset-base',
        'datasets': ALL_DATASETS,      # All datasets index -> run on ALL datasets (incl. SQuAD)
        'alpha': 1.1,
    },
    'dpr-single-nq': {
        'local_index_dir': faiss_single_nq_root,
        'encoder_name': 'facebook/dpr-question_encoder-single-nq-base',
        'datasets': NQ_ONLY_DATASETS,  # NQ-only index -> run ONLY on NQ
        'alpha': 1.1,
    },
}

TOP_K_RETRIEVE = 1000
CHUNK_SIZE = 50_000

## 3. Exact Pyserini CLI Conversion & Evaluation Helper (`-X utf8`)

In [3]:
def run_cmd(cmd_list):
    print('>>', ' '.join(str(x) for x in cmd_list))
    env = {**os.environ, 'PYTHONUTF8': '1'}
    proc = subprocess.run(cmd_list, capture_output=True, text=True, encoding='utf-8', errors='replace', env=env)
    combined = (proc.stdout or '') + '\n' + (proc.stderr or '')
    if combined.strip():
        print(combined.strip())
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd_list, output=proc.stdout, stderr=proc.stderr)
    return combined


def convert_and_eval_trec(trec_path: Path, topics_name: str, use_regex: bool = False):
    """Runs convert_trec_run_to_dpr_retrieval_run and evaluate_dpr_retrieval with UTF-8 forced on Windows."""
    json_path = trec_path.with_suffix('.json')

    if not json_path.exists():
        run_cmd([
            py, '-X', 'utf8', '-m', 'pyserini.eval.convert_trec_run_to_dpr_retrieval_run',
            '--index', root,
            '--topics', topics_name,
            '--input', str(trec_path),
            '--output', str(json_path),
        ])

    eval_cmd = [
        py, '-X', 'utf8', '-m', 'pyserini.eval.evaluate_dpr_retrieval',
        '--retrieval', str(json_path),
        '--topk', '20', '100',
    ]
    if use_regex:
        eval_cmd.append('--regex')

    out_text = run_cmd(eval_cmd)
    m20 = re.search(r'Top20\s+accuracy:\s*([0-9.]+)', out_text)
    m100 = re.search(r'Top100\s+accuracy:\s*([0-9.]+)', out_text)
    if not m20 or not m100:
        raise RuntimeError(f'Failed to parse Top20/Top100 accuracy from output:\n{out_text}')

    return {
        'Top-20': round(float(m20.group(1)) * 100, 2),
        'Top-100': round(float(m100.group(1)) * 100, 2),
    }

## 4. Run BM25 Across All 5 Datasets Using `pyserini.search.lucene` & Local `root`

In [5]:
bm25_records = []

for ds_key, ds_cfg in ALL_DATASETS.items():
    topics_name = ds_cfg['topics']
    tag = ds_cfg['tag']
    trec_out = RUNS_DIR / f'run.dpr.{tag}.bm25.trec'

    if not trec_out.exists():
        run_cmd([
            py, '-m', 'pyserini.search.lucene',
            '--index', root,
            '--topics', topics_name,
            '--output', str(trec_out),
            '--hits', str(TOP_K_RETRIEVE),
            '--threads', '8',
            '--batch-size', '128',
            '--bm25',
        ])
    else:
        print(f'[Skip] Existing BM25 run found: {trec_out}')

    scores = convert_and_eval_trec(trec_out, topics_name, use_regex=ds_cfg['regex'])
    bm25_records.append({
        'Dataset': ds_key.upper(),
        'Retriever': 'BM25',
        'Index': Path(root).name,
        'Top-20': scores['Top-20'],
        'Top-100': scores['Top-100'],
    })

display(pd.DataFrame(bm25_records))

[Skip] Existing BM25 run found: runs\run.dpr.nq-test.bm25.trec
>> c:\Users\Dwija Patel\dwija\sem5\indexing\winenv\Scripts\python.exe -X utf8 -m pyserini.eval.evaluate_dpr_retrieval --retrieval runs\run.dpr.nq-test.bm25.json --topk 20 100
Top20	accuracy: 0.6294
Top100	accuracy: 0.7825


100%|██████████| 3610/3610 [00:00<00:00, 202171.57it/s]
[Skip] Existing BM25 run found: runs\run.dpr.trivia-test.bm25.trec
>> c:\Users\Dwija Patel\dwija\sem5\indexing\winenv\Scripts\python.exe -X utf8 -m pyserini.eval.evaluate_dpr_retrieval --retrieval runs\run.dpr.trivia-test.bm25.json --topk 20 100
Top20	accuracy: 0.7641
Top100	accuracy: 0.8315


100%|██████████| 11313/11313 [00:00<00:00, 75979.58it/s]
[Skip] Existing BM25 run found: runs\run.dpr.wq-test.bm25.trec
>> c:\Users\Dwija Patel\dwija\sem5\indexing\winenv\Scripts\python.exe -X utf8 -m pyserini.eval.evaluate_dpr_retrieval --retrieval runs\run.dpr.wq-test.bm25.json --topk 20 100
Top20	accuracy: 0.6240
Top100	accuracy: 0.7549


100%|██████████| 2

,Dataset,Retriever,Index,Top-20,Top-100
0,NQ,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,62.94,78.25
1,TRIVIA,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,76.41,83.15
2,WQ,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,62.40,75.49
3,CURATED,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,80.69,89.91
4,SQUAD,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,71.09,81.84


## 5. Low-RAM `np.memmap` Chunked Exact Inner-Product Search on Local FAISS Indexes

In [7]:
def open_local_faiss_memmap(local_index_dir: str):
    """Memory-maps your local FAISS IndexFlatIP file without loading 64 GB into RAM."""
    index_path = Path(local_index_dir)
    candidates = sorted(index_path.glob('*.faiss')) or [
        p for p in index_path.iterdir() if p.is_file() and p.name not in {'docid', 'README.md'}
    ]
    if not candidates:
        raise FileNotFoundError(f'No FAISS binary found inside {local_index_dir}')
    faiss_file = max(candidates, key=lambda p: p.stat().st_size)

    with open(faiss_file, 'rb') as f:
        fourcc = f.read(4)
        if fourcc != b'IxFI':
            raise ValueError(f'Expected IndexFlatIP (IxFI), got {fourcc} in {faiss_file}')
        dim = struct.unpack('<i', f.read(4))[0]
        ntotal = struct.unpack('<q', f.read(8))[0]
        _dummy1 = struct.unpack('<q', f.read(8))[0]
        _dummy2 = struct.unpack('<q', f.read(8))[0]
        _is_trained = struct.unpack('<?', f.read(1))[0]
        _metric_type = struct.unpack('<i', f.read(4))[0]
        vec_count = struct.unpack('<Q', f.read(8))[0]
        data_offset = f.tell()

    assert vec_count == ntotal * dim, f'Header mismatch: {vec_count} != {ntotal}*{dim}'

    with open(index_path / 'docid', 'r', encoding='utf-8') as f:
        docids = [line.strip() for line in f]
    assert len(docids) == ntotal, f'Docid count ({len(docids)}) != ntotal ({ntotal})'

    vectors = np.memmap(
        faiss_file, dtype=np.float32, mode='r', offset=data_offset, shape=(ntotal, dim)
    )
    print(f'Memory-mapped local index {faiss_file.name}: shape={vectors.shape}')
    return vectors, docids


@torch.no_grad()
def encode_queries(topics_dict: dict, encoder_name: str, batch_size: int = 64):
    tokenizer = DPRQuestionEncoderTokenizer.from_pretrained(encoder_name)
    model = DPRQuestionEncoder.from_pretrained(encoder_name).to(DEVICE)
    model.eval()

    qids = list(topics_dict.keys())
    queries = [topics_dict[qid]['title'] for qid in qids]
    all_embs = []

    for start in range(0, len(queries), batch_size):
        batch_q = queries[start:start + batch_size]
        inputs = tokenizer(
            batch_q, padding=True, truncation=True, max_length=256, return_tensors='pt'
        ).to(DEVICE)
        emb = model(**inputs).pooler_output.cpu().numpy().astype(np.float32)
        all_embs.append(emb)

    del model
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return qids, np.vstack(all_embs)


def chunked_exact_ip_search(
    passage_memmap: np.memmap,
    docids: list,
    qids: list,
    query_embs: np.ndarray,
    trec_out_path: Path,
    ckpt_path: Path,
    run_tag: str,
    topk: int = 1000,
    chunk_size: int = 50_000,
    ckpt_every_chunks: int = 20,
):
    if trec_out_path.exists():
        print(f'[Skip] Existing TREC run found: {trec_out_path}')
        return

    ntotal = passage_memmap.shape[0]
    num_q = len(qids)
    start_row = 0

    if ckpt_path.exists():
        ckpt = np.load(ckpt_path)
        start_row = int(ckpt['next_row'])
        best_scores = ckpt['best_scores']
        best_indices = ckpt['best_indices']
        print(f'Resuming {run_tag} from row {start_row:,}/{ntotal:,}')
    else:
        best_scores = np.full((num_q, topk), -np.inf, dtype=np.float32)
        best_indices = np.full((num_q, topk), -1, dtype=np.int64)

    q_T = np.ascontiguousarray(query_embs.T)
    total_chunks = (ntotal - start_row + chunk_size - 1) // chunk_size

    for chunk_idx, row_start in enumerate(
        tqdm(range(start_row, ntotal, chunk_size), total=total_chunks, desc=run_tag), start=1
    ):
        row_end = min(row_start + chunk_size, ntotal)
        chunk = np.asarray(passage_memmap[row_start:row_end], dtype=np.float32)
        scores_T = (chunk @ q_T).T

        k_chunk = min(topk, row_end - row_start)
        part_idx = np.argpartition(-scores_T, kth=k_chunk - 1, axis=1)[:, :k_chunk]
        part_scores = np.take_along_axis(scores_T, part_idx, axis=1)
        global_idx = part_idx + row_start

        merged_scores = np.concatenate([best_scores, part_scores], axis=1)
        merged_indices = np.concatenate([best_indices, global_idx], axis=1)

        top_merge = np.argpartition(-merged_scores, kth=topk - 1, axis=1)[:, :topk]
        best_scores = np.take_along_axis(merged_scores, top_merge, axis=1)
        best_indices = np.take_along_axis(merged_indices, top_merge, axis=1)

        if chunk_idx % ckpt_every_chunks == 0:
            np.savez(
                ckpt_path,
                next_row=np.array(row_end, dtype=np.int64),
                best_scores=best_scores,
                best_indices=best_indices,
            )

    order = np.argsort(-best_scores, axis=1)
    sorted_scores = np.take_along_axis(best_scores, order, axis=1)
    sorted_indices = np.take_along_axis(best_indices, order, axis=1)

    with open(trec_out_path, 'w', encoding='utf-8') as f:
        for qi, qid in enumerate(qids):
            for rank in range(topk):
                doc_idx = int(sorted_indices[qi, rank])
                if doc_idx < 0:
                    continue
                f.write(f'{qid} Q0 {docids[doc_idx]} {rank + 1} {sorted_scores[qi, rank]:.6f} {run_tag}\n')

    if ckpt_path.exists():
        ckpt_path.unlink()
    print(f'[Done] Wrote TREC run: {trec_out_path}')


def load_trec_scores(trec_path: Path):
    runs = {}
    with open(trec_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 6:
                qid, _, docid, _, score, _ = parts[:6]
                runs.setdefault(qid, {})[docid] = float(score)
    return runs


def build_hybrid_trec_run(dense_trec: Path, bm25_trec: Path, hybrid_out: Path, alpha: float, run_tag: str, topk: int = 1000):
    if hybrid_out.exists():
        print(f'[Skip] Existing Hybrid run found: {hybrid_out}')
        return
    dense_runs = load_trec_scores(dense_trec)
    bm25_runs = load_trec_scores(bm25_trec)
    with open(hybrid_out, 'w', encoding='utf-8') as f:
        for qid, d_docs in dense_runs.items():
            b_docs = bm25_runs.get(qid, {})
            all_docids = set(d_docs.keys()) | set(b_docs.keys())
            scored = [(docid, d_docs.get(docid, 0.0) + alpha * b_docs.get(docid, 0.0)) for docid in all_docids]
            scored.sort(key=lambda x: x[1], reverse=True)
            for rank, (docid, score) in enumerate(scored[:topk], start=1):
                f.write(f'{qid} Q0 {docid} {rank} {score:.6f} {run_tag}\n')
    print(f'[Done] Wrote Hybrid TREC run: {hybrid_out}')

## 6. Run Dense & Hybrid Retrieval + CLI Evaluation

- **`dpr-multi` (`faiss_multi_root`)** runs across **all 5 datasets** (NQ, TriviaQA, WebQuestions, CuratedTREC, **SQuAD**).
- **`dpr-single-nq` (`faiss_single_nq_root`)** runs **only on NQ** (`dpr-nq-test`).

In [8]:
all_records = list(bm25_records)

for mode_key, cfg in DENSE_CONFIGS.items():
    print(f'\n================ Running {mode_key} on {cfg["local_index_dir"]} ================')
    passage_memmap, docids = open_local_faiss_memmap(cfg['local_index_dir'])

    for ds_key, ds_cfg in cfg['datasets'].items():
        topics_name = ds_cfg['topics']
        tag = ds_cfg['tag']
        topics = get_topics(topics_name)
        qids, query_embs = encode_queries(topics, cfg['encoder_name'])

        dense_trec = RUNS_DIR / f'run.dpr.{tag}.{mode_key}.trec'
        ckpt_file = CKPT_DIR / f'ckpt.dpr.{tag}.{mode_key}.npz'
        chunked_exact_ip_search(
            passage_memmap=passage_memmap,
            docids=docids,
            qids=qids,
            query_embs=query_embs,
            trec_out_path=dense_trec,
            ckpt_path=ckpt_file,
            run_tag=mode_key,
            topk=TOP_K_RETRIEVE,
            chunk_size=CHUNK_SIZE,
        )

        # Evaluate Dense run using your exact CLI commands
        dense_scores = convert_and_eval_trec(dense_trec, topics_name, use_regex=ds_cfg['regex'])
        all_records.append({
            'Dataset': ds_key.upper(),
            'Retriever': 'DPR-Multi' if mode_key == 'multi' else 'DPR-Single (NQ)',
            'Index': Path(cfg['local_index_dir']).name,
            'Top-20': dense_scores['Top-20'],
            'Top-100': dense_scores['Top-100'],
        })

        # Build & evaluate Hybrid (BM25 + Dense) run using your exact CLI commands
        bm25_trec = RUNS_DIR / f'run.dpr.{tag}.bm25.trec'
        hybrid_tag = 'hybrid-multi' if mode_key == 'multi' else 'hybrid-single-nq'
        hybrid_trec = RUNS_DIR / f'run.hybrid.{tag}.{hybrid_tag}.trec'
        build_hybrid_trec_run(
            dense_trec=dense_trec,
            bm25_trec=bm25_trec,
            hybrid_out=hybrid_trec,
            alpha=cfg['alpha'],
            run_tag=hybrid_tag,
            topk=TOP_K_RETRIEVE,
        )

        hybrid_scores = convert_and_eval_trec(hybrid_trec, topics_name, use_regex=ds_cfg['regex'])
        all_records.append({
            'Dataset': ds_key.upper(),
            'Retriever': 'BM25 + DPR-Multi' if mode_key == 'multi' else 'BM25 + DPR-Single (NQ)',
            'Index': Path(cfg['local_index_dir']).name,
            'Top-20': hybrid_scores['Top-20'],
            'Top-100': hybrid_scores['Top-100'],
        })

    del passage_memmap

results_df = pd.DataFrame(all_records)
results_df.to_csv(RUNS_DIR / 'dpr_final_table_results.csv', index=False)
pivot_df = results_df.pivot(index='Retriever', columns='Dataset', values=['Top-20', 'Top-100'])
display(results_df)
display(pivot_df)


================ Running multi on C:\Users\Dwija Patel\.cache\pyserini\indexes\faiss.wikipedia-dpr-100w.dpr_multi.20200127.f403c3 ================
Memory-mapped local index index: shape=(21015324, 768)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] DPRQuestionEncoder LOAD REPORT from: facebook/dpr-question_encoder-multiset-base
Key                                             | Status     |  | 
------------------------------------------------+------------+--+-
question_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 
question_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Skip] Existing TREC run found: runs\run.dpr.nq-test.multi.trec
>> c:\Users\Dwija Patel\dwija\sem5\indexing\winenv\Scripts\python.exe -X utf8 -m pyserini.eval.evaluate_dpr_retrieval --retrieval runs\run.dpr.nq-test.multi.json --topk 20 100
Top20	accuracy: 0.7947
Top100	accuracy: 0.8609


100%|██████████| 3610/3610 [00:00<00:00, 289251.29it/s]
[Skip] Existing Hybrid run found: runs\run.hybrid.nq-test.hybrid-multi.trec
>> c:\Users\Dwija Patel\dwija\sem5\indexing\winenv\Scripts\python.exe -X utf8 -m pyserini.eval.evaluate_dpr_retrieval --retrieval runs\run.hybrid.nq-test.hybrid-multi.json --topk 20 100
Top20	accuracy: 0.8188
Top100	accuracy: 0.8609


100%|██████████| 3610/3610 [00:00<00:00, 250881.27it/s]


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] DPRQuestionEncoder LOAD REPORT from: facebook/dpr-question_encoder-multiset-base
Key                                             | Status     |  | 
------------------------------------------------+------------+--+-
question_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 
question_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


... (15 outputs left out)

[transformers] DPRQuestionEncoder LOAD REPORT from: facebook/dpr-question_encoder-single-nq-base
Key                                             | Status     |  | 
------------------------------------------------+------------+--+-
question_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 
question_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Skip] Existing TREC run found: runs\run.dpr.nq-test.dpr-single-nq.trec
>> c:\Users\Dwija Patel\dwija\sem5\indexing\winenv\Scripts\python.exe -X utf8 -m pyserini.eval.evaluate_dpr_retrieval --retrieval runs\run.dpr.nq-test.dpr-single-nq.json --topk 20 100
Top20	accuracy: 0.8006
Top100	accuracy: 0.8609


100%|██████████| 3610/3610 [00:00<00:00, 317730.30it/s]
[Skip] Existing Hybrid run found: runs\run.hybrid.nq-test.hybrid-single-nq.trec
>> c:\Users\Dwija Patel\dwija\sem5\indexing\winenv\Scripts\python.exe -X utf8 -m pyserini.eval.evaluate_dpr_retrieval --retrieval runs\run.hybrid.nq-test.hybrid-single-nq.json --topk 20 100
Top20	accuracy: 0.8186
Top100	accuracy: 0.8609


100%|██████████| 3610/3610 [00:00<00:00, 369953.03it/s]


,Dataset,Retriever,Index,Top-20,Top-100
0,NQ,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,62.94,78.25
1,TRIVIA,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,76.41,83.15
2,WQ,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,62.40,75.49
3,CURATED,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,80.69,89.91
4,SQUAD,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,71.09,81.84
5,NQ,DPR-Multi,faiss.wikipedia-dpr-100w.dpr_multi.20200127.f4...,79.47,86.09
6,NQ,BM25 + DPR-Multi,faiss.wikipedia-dpr-100w.dpr_multi.20200127.f4...,81.88,86.09
7,TRIVIA,DPR-Multi,faiss.wikipedia-dpr-100w.dpr_multi.20200127.f4...,78.87,84.79
8,TRIVIA,BM25 + DPR-Multi,faiss.wikipedia-dpr-100w.dpr_multi.20200127.f4...,81.05,84.79
9,WQ,DPR-Multi,faiss.wikipedia-dpr-100w.dpr_multi.20200127.f4...,75.05,82.97


Top-20                             Top-100         \
Dataset                CURATED     NQ  SQUAD TRIVIA     WQ CURATED     NQ   
Retriever                                                                   
BM25                     80.69  62.94  71.09  76.41  62.40   89.91  78.25   
BM25 + DPR-Multi         89.19  81.88  67.28  81.05  76.67   93.37  86.09   
BM25 + DPR-Single (NQ)     NaN  81.86    NaN    NaN    NaN     NaN  86.09   
DPR-Multi                88.76  79.47  51.99  78.87  75.05   93.37  86.09   
DPR-Single (NQ)            NaN  80.06    NaN    NaN    NaN     NaN  86.09   

                                             
Dataset                 SQUAD TRIVIA     WQ  
Retriever                                    
BM25                    81.84  83.15  75.49  
BM25 + DPR-Multi        75.78  84.79  82.97  
BM25 + DPR-Single (NQ)    NaN    NaN    NaN  
DPR-Multi               67.73  84.79  82.97  
DPR-Single (NQ)           NaN    NaN    NaN

## 7. Extension 1 - RRF

In [4]:
import ast
import json
from pathlib import Path
import pandas as pd
from pyserini.search import get_topics
from pyserini.search.lucene import LuceneSearcher
from pyserini.eval.evaluate_dpr_retrieval import has_answers, SimpleTokenizer

lucene_searcher = LuceneSearcher(root)
simple_tok = SimpleTokenizer()

def load_trec_top100(trec_path: Path, max_k: int = 100):
    runs = {}
    with open(trec_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 6:
                qid, _, docid, _, score, _ = parts[:6]
                lst = runs.setdefault(qid, [])
                if len(lst) < max_k:
                    lst.append((docid, float(score)))
    return runs

def fuse_rrf(bm25_runs: dict, dpr_runs: dict, k: int = 60, topk: int = 100):
    fused = {}
    for qid in dpr_runs:
        scores = {}
        for r, (docid, _) in enumerate(bm25_runs.get(qid, []), start=1):
            scores[docid] = scores.get(docid, 0.0) + 1.0 / (k + r)
        for r, (docid, _) in enumerate(dpr_runs.get(qid, []), start=1):
            scores[docid] = scores.get(docid, 0.0) + 1.0 / (k + r)
        fused[qid] = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:topk]
    return fused

def eval_recall_and_mrr(runs: dict, topics: dict, use_regex: bool = False, passage_cache: dict = None, qids_subset=None):
    if passage_cache is None:
        passage_cache = {}
    qids = qids_subset if qids_subset is not None else [q for q in topics if str(q) in runs]
    r5 = r20 = r100 = mrr_sum = 0.0

    for qid in qids:
        q_str = str(qid)
        raw_ans = topics[qid]['answers']
        answers = ast.literal_eval(raw_ans) if isinstance(raw_ans, str) and raw_ans.startswith('[') else (
            raw_ans if isinstance(raw_ans, list) else [str(raw_ans)]
        )
        cands = runs.get(q_str, [])[:100]
        first_hit_rank = None

        for rank, (docid, _) in enumerate(cands, start=1):
            if docid not in passage_cache:
                doc = lucene_searcher.doc(str(docid))
                passage_cache[docid] = json.loads(doc.raw()).get('contents', '') if doc else ''
            text = passage_cache[docid]
            if has_answers(text, answers, simple_tok, regex=use_regex):
                first_hit_rank = rank
                break

        if first_hit_rank is not None:
            if first_hit_rank <= 5:   r5 += 1
            if first_hit_rank <= 20:  r20 += 1
            if first_hit_rank <= 100: r100 += 1
            mrr_sum += 1.0 / first_hit_rank

    n = max(len(qids), 1)
    return {
        'Recall@5': round(100.0 * r5 / n, 2),
        'Recall@20': round(100.0 * r20 / n, 2),
        'Recall@100': round(100.0 * r100 / n, 2),
        'MRR': round(mrr_sum / n, 4),
    }

In [ ]:
# Load full runs for all 5 datasets & define a small tuning slice
passage_cache = {}
datasets_data = {}

for ds_key, ds_cfg in ALL_DATASETS.items():
    tag, topics_name, use_reg = ds_cfg['tag'], ds_cfg['topics'], ds_cfg['regex']
    topics = get_topics(topics_name)
    bm25_r = load_trec_top100(RUNS_DIR / f'run.dpr.{tag}.bm25.trec', max_k=1000)
    dpr_r  = load_trec_top100(RUNS_DIR / f'run.dpr.{tag}.multi.trec', max_k=1000)
    lin_r  = load_trec_top100(RUNS_DIR / f'run.hybrid.{tag}.hybrid-multi.trec', max_k=100)

    all_qids = [q for q in topics if str(q) in dpr_r]
    n_tune = min(300, max(100, int(0.2 * len(all_qids))))
    datasets_data[ds_key] = {
        'tag': tag,
        'topics': topics,
        'regex': use_reg,
        'tune_qids': all_qids[:n_tune],   # Subset used ONLY to pick k
        'bm25': bm25_r,
        'dpr': dpr_r,
        'linear': lin_r,
    }

# Tune RRF constant k across all 5 tuning slices
k_candidates = [1, 10, 20, 40, 60, 100]
tune_rows = []

for k_val in k_candidates:
    for ds_key, d in datasets_data.items():
        rrf_r = fuse_rrf(d['bm25'], d['dpr'], k=k_val, topk=100)
        m = eval_recall_and_mrr(
            rrf_r, d['topics'], use_regex=d['regex'],
            passage_cache=passage_cache, qids_subset=d['tune_qids']
        )
        tune_rows.append({'RRF k': k_val, 'Dataset': ds_key.upper(), **m})

tune_df = pd.DataFrame(tune_rows)
tune_mrr_pivot = tune_df.pivot(index='RRF k', columns='Dataset', values='MRR')
tune_mrr_pivot['Macro-Avg MRR'] = tune_mrr_pivot.mean(axis=1).round(4)

best_k = int(tune_mrr_pivot['Macro-Avg MRR'].idxmax())
print(f'Selected Global RRF k = {best_k}')
display(tune_mrr_pivot)

# Evaluate on the ENTIRE test set for all 5 datasets 
full_rows = []
for ds_key, d in datasets_data.items():
    rrf_full = fuse_rrf(d['bm25'], d['dpr'], k=best_k, topk=100)

    # Save full RRF TREC file for downstream Extension 2 (Cross-Encoder) & Table 4 Reader
    rrf_out = RUNS_DIR / f"run.dpr.{d['tag']}.rrf-k{best_k}.trec"
    with open(rrf_out, 'w', encoding='utf-8') as f:
        for qid, docs in rrf_full.items():
            for rank, (docid, sc) in enumerate(docs, start=1):
                f.write(f'{qid} Q0 {docid} {rank} {sc:.8f} rrf-k{best_k}\n')

    for method_label, run_dict in [
        ('DPR', d['dpr']),
        ('BM25', d['bm25']),
        ('DPR + BM25 (Linear, λ=1.1)', d['linear']),
        (f'DPR + BM25 + RRF (k={best_k})', rrf_full),
    ]:
        # qids_subset=None evaluates on the ENTIRE dataset
        m = eval_recall_and_mrr(
            run_dict, d['topics'], use_regex=d['regex'],
            passage_cache=passage_cache, qids_subset=None
        )
        full_rows.append({'Dataset': ds_key.upper(), 'Method': method_label, **m})

final_full_df = pd.DataFrame(full_rows)
final_full_df.to_csv(RUNS_DIR / 'extension1_rrf_full_dataset_metrics.csv', index=False)
display(final_full_df)

Selected Global RRF k = 20


Dataset,CURATED,NQ,SQUAD,TRIVIA,WQ,Macro-Avg MRR
RRF k,,,,,,
1,0.6955,0.4931,0.4278,0.6115,0.5249,0.5506
10,0.6767,0.5265,0.4350,0.6180,0.5352,0.5583
20,0.6804,0.5335,0.4383,0.6204,0.5332,0.5612
40,0.6953,0.5240,0.4228,0.6250,0.5219,0.5578
60,0.6896,0.5298,0.4115,0.6336,0.5211,0.5571
100,0.6886,0.5288,0.4007,0.6342,0.5169,0.5538


,Dataset,Method,Recall@5,Recall@20,Recall@100,MRR
0,NQ,DPR,67.31,79.61,86.26,0.5485
1,NQ,BM25,44.63,63.91,78.81,0.3317
2,NQ,"DPR + BM25 (Linear, λ=1.1)",71.63,82.02,86.26,0.5903
3,NQ,DPR + BM25 + RRF (k=20),68.95,81.52,88.25,0.5406
4,TRIVIA,DPR,70.14,79.15,84.96,0.6086
5,TRIVIA,BM25,67.44,77.28,83.88,0.5686
6,TRIVIA,"DPR + BM25 (Linear, λ=1.1)",75.50,81.34,84.96,0.6779
7,TRIVIA,DPR + BM25 + RRF (k=20),75.82,82.70,86.88,0.6703
8,WQ,DPR,64.32,75.39,83.42,0.5362
9,WQ,BM25,42.81,63.24,76.43,0.3051


In [8]:
# Reuse best_k selected in the previous cell (no re-tuning)
print(f'Using previously selected RRF k = {best_k} for DPR-Single (NQ)')

nq_topics = get_topics('dpr-nq-test')
bm25_nq = load_trec_top100(RUNS_DIR / 'run.dpr.nq-test.bm25.trec', max_k=1000)
dpr_single_nq = load_trec_top100(RUNS_DIR / 'run.dpr.nq-test.dpr-single-nq.trec', max_k=1000)
lin_single_nq = load_trec_top100(RUNS_DIR / 'run.hybrid.nq-test.hybrid-single-nq.trec', max_k=100)

rrf_single_full = fuse_rrf(bm25_nq, dpr_single_nq, k=best_k, topk=100)

# Save TREC run using the same best_k
rrf_single_out = RUNS_DIR / f'run.dpr.nq-test.rrf-single-nq-k{best_k}.trec'
with open(rrf_single_out, 'w', encoding='utf-8') as f:
    for qid, docs in rrf_single_full.items():
        for rank, (docid, sc) in enumerate(docs, start=1):
            f.write(f'{qid} Q0 {docid} {rank} {sc:.8f} rrf-single-nq-k{best_k}\n')

single_full_rows = []
for method_label, run_dict in [
    ('DPR-Single (NQ)', dpr_single_nq),
    ('BM25', bm25_nq),
    ('DPR-Single + BM25 (Linear, λ=1.3)', lin_single_nq),
    (f'DPR-Single + BM25 + RRF (k={best_k})', rrf_single_full),
]:
    m = eval_recall_and_mrr(
        run_dict, nq_topics, use_regex=False,
        passage_cache=passage_cache, qids_subset=None
    )
    single_full_rows.append({'Dataset': 'NQ', 'Method': method_label, **m})

single_final_df = pd.DataFrame(single_full_rows)
single_final_df.to_csv(RUNS_DIR / 'extension1_rrf_single_nq_metrics.csv', index=False)
display(single_final_df)

Using previously selected RRF k = 20 for DPR-Single (NQ)


,Dataset,Method,Recall@5,Recall@20,Recall@100,MRR
0,NQ,DPR-Single (NQ),68.56,80.25,86.23,0.5654
1,NQ,BM25,44.63,63.91,78.81,0.3317
2,NQ,"DPR-Single + BM25 (Linear, λ=1.3)",71.25,82.08,86.23,0.5916
3,NQ,DPR-Single + BM25 + RRF (k=20),69.39,82.05,88.42,0.5446


## 8. Extension 2 - Post Retreival Cross Encoder

In [7]:
import os
import ast
import time
import json
from pathlib import Path
import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from pyserini.search import get_topics
from pyserini.search.lucene import LuceneSearcher
from pyserini.eval.evaluate_dpr_retrieval import has_answers, SimpleTokenizer

# --- Self-contained setup ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RUNS_DIR = Path('runs')
RUNS_DIR.mkdir(parents=True, exist_ok=True)

INDEX_CACHE_DIR = Path(os.path.expanduser('~')) / '.cache' / 'pyserini' / 'indexes'
lucene_matches = sorted([p for p in INDEX_CACHE_DIR.glob('lucene-inverted.wikipedia-dpr-100w*') if p.is_dir()])
if not lucene_matches:
    raise FileNotFoundError(f'Local Lucene index not found in {INDEX_CACHE_DIR}')
root = str(lucene_matches[-1])

ALL_DATASETS = {
    'nq':      {'topics': 'dpr-nq-test',      'tag': 'nq-test',      'regex': False},
    'trivia':  {'topics': 'dpr-trivia-test',  'tag': 'trivia-test',  'regex': False},
    'wq':      {'topics': 'dpr-wq-test',      'tag': 'wq-test',      'regex': False},
    'curated': {'topics': 'dpr-curated-test', 'tag': 'curated-test', 'regex': True},
    'squad':   {'topics': 'dpr-squad-test',   'tag': 'squad-test',   'regex': False},
}

lucene_searcher = LuceneSearcher(root)
simple_tok = SimpleTokenizer()
passage_cache = {}

def load_trec_topk(trec_path: Path, max_k: int = 50):
    runs = {}
    with open(trec_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 6:
                qid, _, docid, _, score, _ = parts[:6]
                lst = runs.setdefault(qid, [])
                if len(lst) < max_k:
                    lst.append((docid, float(score)))
    return runs

def get_cached_passage(docid: str) -> str:
    if docid not in passage_cache:
        doc = lucene_searcher.doc(str(docid))
        passage_cache[docid] = json.loads(doc.raw()).get('contents', '') if doc else ''
    return passage_cache[docid]

def eval_recall_and_mrr(runs: dict, topics: dict, use_regex: bool = False, qids_subset=None):
    qids = qids_subset if qids_subset is not None else [q for q in topics if str(q) in runs]
    r5 = r20 = r50 = mrr_sum = 0.0

    for qid in qids:
        q_str = str(qid)
        raw_ans = topics[qid]['answers']
        answers = ast.literal_eval(raw_ans) if isinstance(raw_ans, str) and raw_ans.startswith('[') else (
            raw_ans if isinstance(raw_ans, list) else [str(raw_ans)]
        )
        cands = runs.get(q_str, [])[:50]
        first_hit_rank = None

        for rank, (docid, _) in enumerate(cands, start=1):
            text = get_cached_passage(docid)
            if has_answers(text, answers, simple_tok, regex=use_regex):
                first_hit_rank = rank
                break

        if first_hit_rank is not None:
            if first_hit_rank <= 5:  r5 += 1
            if first_hit_rank <= 20: r20 += 1
            if first_hit_rank <= 50: r50 += 1
            mrr_sum += 1.0 / first_hit_rank

    n = max(len(qids), 1)
    return {
        'Recall@5': round(100.0 * r5 / n, 2),
        'Recall@20': round(100.0 * r20 / n, 2),
        'Recall@50': round(100.0 * r50 / n, 2),
        'MRR': round(mrr_sum / n, 4),
    }

# --- Fast CPU Cross-Encoder Setup & Reranking on Original DPR Only ---
CE_MODEL_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
RERANK_TOP_M = 50
CE_BATCH_SIZE = 64
MAX_EVAL_QUERIES = 500    # Same 500 queries compared for both Original DPR and CE-Reranked

ce_tokenizer = AutoTokenizer.from_pretrained(CE_MODEL_NAME)
ce_model = AutoModelForSequenceClassification.from_pretrained(CE_MODEL_NAME).eval()
if DEVICE == 'cpu':
    ce_model = torch.quantization.quantize_dynamic(ce_model, {torch.nn.Linear}, dtype=torch.qint8)
else:
    ce_model = ce_model.to(DEVICE)
print(f'Loaded {CE_MODEL_NAME} on {DEVICE} (top_m={RERANK_TOP_M}, max_queries={MAX_EVAL_QUERIES})')

@torch.no_grad()
def rerank_dpr_with_cross_encoder(
    dpr_runs: dict,
    topics: dict,
    out_trec_path: Path,
    top_m: int = RERANK_TOP_M,
    batch_size: int = CE_BATCH_SIZE,
    max_queries: int = MAX_EVAL_QUERIES,
    run_tag: str = 'dpr-ce-minilm-l6',
):
    qids = [q for q in topics if str(q) in dpr_runs]
    if max_queries is not None:
        qids = qids[:max_queries]

    reranked_runs = {}
    total_rerank_sec = 0.0

    with open(out_trec_path, 'w', encoding='utf-8') as fout:
        for qid in tqdm(qids, desc=f'CE Rerank ({run_tag})'):
            q_str = str(qid)
            q_text = topics[qid]['title']
            cands = dpr_runs[q_str][:top_m]

            docids = [d for d, _ in cands]
            passages = [get_cached_passage(d) for d in docids]

            t0 = time.perf_counter()
            ce_scores = []
            for i in range(0, len(passages), batch_size):
                b_passages = passages[i:i + batch_size]
                enc = ce_tokenizer(
                    [q_text] * len(b_passages),
                    b_passages,
                    padding=True,
                    truncation=True,
                    max_length=192,
                    return_tensors='pt',
                ).to(DEVICE)
                logits = ce_model(**enc).logits.squeeze(-1)
                if logits.ndim == 0:
                    logits = logits.unsqueeze(0)
                ce_scores.extend(logits.cpu().tolist())
            if DEVICE == 'cuda':
                torch.cuda.synchronize()
            total_rerank_sec += (time.perf_counter() - t0)

            ranked = sorted(zip(docids, ce_scores), key=lambda x: x[1], reverse=True)
            reranked_runs[q_str] = ranked
            for rank, (docid, score) in enumerate(ranked, start=1):
                fout.write(f'{q_str} Q0 {docid} {rank} {score:.6f} {run_tag}\n')

    avg_latency_ms = round((total_rerank_sec / max(len(qids), 1)) * 1000.0, 2)
    return reranked_runs, avg_latency_ms, qids

ce_compare_rows = []

# 1. Original DPR-Multi vs. DPR-Multi + Cross-Encoder across all 5 datasets
for ds_key, ds_cfg in ALL_DATASETS.items():
    tag, topics_name, use_reg = ds_cfg['tag'], ds_cfg['topics'], ds_cfg['regex']
    topics = get_topics(topics_name)

    orig_dpr_runs = load_trec_topk(RUNS_DIR / f'run.dpr.{tag}.multi.trec', max_k=RERANK_TOP_M)
    ce_out = RUNS_DIR / f'run.dpr.{tag}.dpr-multi-ce-minilm-l6.trec'

    reranked_runs, lat_ms, eval_qids = rerank_dpr_with_cross_encoder(
        orig_dpr_runs, topics, ce_out, run_tag=f'dpr-multi-ce-{tag}'
    )

    m_orig = eval_recall_and_mrr(orig_dpr_runs, topics, use_regex=use_reg, qids_subset=eval_qids)
    m_ce = eval_recall_and_mrr(reranked_runs, topics, use_regex=use_reg, qids_subset=eval_qids)

    ce_compare_rows.append({
        'Dataset': ds_key.upper(),
        'Method': 'DPR-Multi (Original)',
        'Rerank Latency (ms/q)': 0.0,
        **m_orig,
    })
    ce_compare_rows.append({
        'Dataset': ds_key.upper(),
        'Method': 'DPR-Multi + Cross-Encoder (MiniLM-L6)',
        'Rerank Latency (ms/q)': lat_ms,
        **m_ce,
    })

# 2. Original DPR-Single (NQ) vs. DPR-Single (NQ) + Cross-Encoder on NQ
nq_topics = get_topics('dpr-nq-test')
orig_single_runs = load_trec_topk(RUNS_DIR / 'run.dpr.nq-test.dpr-single-nq.trec', max_k=RERANK_TOP_M)
ce_single_out = RUNS_DIR / 'run.dpr.nq-test.dpr-single-nq-ce-minilm-l6.trec'

reranked_single, lat_ms_s, eval_qids_s = rerank_dpr_with_cross_encoder(
    orig_single_runs, nq_topics, ce_single_out, run_tag='dpr-single-nq-ce'
)

m_orig_s = eval_recall_and_mrr(orig_single_runs, nq_topics, use_regex=False, qids_subset=eval_qids_s)
m_ce_s = eval_recall_and_mrr(reranked_single, nq_topics, use_regex=False, qids_subset=eval_qids_s)

ce_compare_rows.append({
    'Dataset': 'NQ',
    'Method': 'DPR-Single (Original)',
    'Rerank Latency (ms/q)': 0.0,
    **m_orig_s,
})
ce_compare_rows.append({
    'Dataset': 'NQ',
    'Method': 'DPR-Single + Cross-Encoder (MiniLM-L6)',
    'Rerank Latency (ms/q)': lat_ms_s,
    **m_ce_s,
})

del ce_model
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

ce_compare_df = pd.DataFrame(ce_compare_rows)
ce_compare_df.to_csv(RUNS_DIR / 'extension2_dpr_only_ce_comparison.csv', index=False)
display(ce_compare_df)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

C:\Users\Dwija Patel\AppData\Local\Temp\ipykernel_29488\1331750981.py:97: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  ce_model = torch.quantization.quantize_dynamic(ce_model, {torch.nn.Linear}, dtype=torch.qint8)


Loaded cross-encoder/ms-marco-MiniLM-L-6-v2 on cpu (top_m=50, max_queries=500)


CE Rerank (dpr-multi-ce-nq-test):   0%|          | 0/500 [00:00<?, ?it/s]

CE Rerank (dpr-multi-ce-trivia-test):   0%|          | 0/500 [00:00<?, ?it/s]

CE Rerank (dpr-multi-ce-wq-test):   0%|          | 0/500 [00:00<?, ?it/s]

CE Rerank (dpr-multi-ce-curated-test):   0%|          | 0/500 [00:00<?, ?it/s]

CE Rerank (dpr-multi-ce-squad-test):   0%|          | 0/500 [00:00<?, ?it/s]

CE Rerank (dpr-single-nq-ce):   0%|          | 0/500 [00:00<?, ?it/s]

,Dataset,Method,Rerank Latency (ms/q),Recall@5,Recall@20,Recall@50,MRR
0,NQ,DPR-Multi (Original),0.00,65.4,79.0,83.8,0.5261
1,NQ,DPR-Multi + Cross-Encoder (MiniLM-L6),1438.82,72.2,80.8,83.8,0.5814
2,TRIVIA,DPR-Multi (Original),0.00,68.2,79.6,83.2,0.5835
3,TRIVIA,DPR-Multi + Cross-Encoder (MiniLM-L6),1592.50,73.6,81.4,83.2,0.6730
4,WQ,DPR-Multi (Original),0.00,65.4,74.0,80.0,0.5459
5,WQ,DPR-Multi + Cross-Encoder (MiniLM-L6),1511.07,66.6,75.6,80.0,0.5525
6,CURATED,DPR-Multi (Original),0.00,80.8,88.8,91.0,0.7203
7,CURATED,DPR-Multi + Cross-Encoder (MiniLM-L6),1546.61,86.0,90.6,91.0,0.7713
8,SQUAD,DPR-Multi (Original),0.00,35.6,52.6,61.6,0.2729
9,SQUAD,DPR-Multi + Cross-Encoder (MiniLM-L6),1588.26,48.2,56.8,61.6,0.3933


In [11]:
def get_cached_passage(docid: str) -> str:
    if docid not in passage_cache:
        doc = lucene_searcher.doc(str(docid))
        passage_cache[docid] = json.loads(doc.raw()).get('contents', '') if doc else ''
    val = passage_cache[docid]
    return f'{val[0]}\n{val[1]}' if isinstance(val, tuple) else val

def minmax_norm(scores: list) -> list:
    lo, hi = min(scores), max(scores)
    return [0.5] * len(scores) if (hi - lo) < 1e-9 else [(s - lo) / (hi - lo) for s in scores]

def blend_dpr_and_ce(dpr_runs: dict, ce_runs: dict, beta: float = 0.8, top_m: int = 50):
    blended_runs = {}
    for qid, dpr_cands in dpr_runs.items():
        head = dpr_cands[:top_m]
        tail = dpr_cands[top_m:]
        ce_map = dict(ce_runs.get(qid, []))

        docids = [d for d, _ in head]
        dpr_norm = minmax_norm([sc for _, sc in head])
        ce_norm  = minmax_norm([ce_map.get(d, 0.0) for d in docids])

        rescored = [
            (d, (1.0 - beta) * s_dpr + beta * s_ce)
            for d, s_dpr, s_ce in zip(docids, dpr_norm, ce_norm)
        ]
        rescored.sort(key=lambda x: x[1], reverse=True)
        min_sc = rescored[-1][1] if rescored else 0.0
        blended_runs[qid] = rescored + [(d, min_sc - 1.0 - i) for i, (d, _) in enumerate(tail)]
    return blended_runs

# Tune beta and M on NQ tuning slice (first 200 queries of your 500 evaluated queries)
nq_topics = get_topics('dpr-nq-test')
dpr_nq = load_trec_topk(RUNS_DIR / 'run.dpr.nq-test.multi.trec', max_k=50)
ce_nq  = load_trec_topk(RUNS_DIR / 'run.dpr.nq-test.dpr-multi-ce-minilm-l6.trec', max_k=50)
tune_qids = [q for q in nq_topics if str(q) in ce_nq][:200]

tune_ce_rows = []
for m_val in [10, 20, 50]:
    for beta_val in [0.0, 0.5, 0.7, 0.85, 1.0]:
        blended = blend_dpr_and_ce(dpr_nq, ce_nq, beta=beta_val, top_m=m_val)
        metrics = eval_recall_and_mrr(blended, nq_topics, use_regex=False, qids_subset=tune_qids)
        tune_ce_rows.append({'Top-M': m_val, 'Beta (CE weight)': beta_val, **metrics})

ce_tune_df = pd.DataFrame(tune_ce_rows)
display(ce_tune_df.sort_values(by='MRR', ascending=False))

,Top-M,Beta (CE weight),Recall@5,Recall@20,Recall@50,MRR
13,50,0.85,77.0,82.5,85.5,0.6425
12,50,0.70,76.0,82.5,85.5,0.6244
8,20,0.85,74.0,78.5,85.5,0.6229
7,20,0.70,73.5,78.5,85.5,0.6158
3,10,0.85,72.5,78.5,85.5,0.6081
4,10,1.00,72.5,78.5,85.5,0.6075
14,50,1.00,76.0,83.0,85.5,0.6074
9,20,1.00,72.5,78.5,85.5,0.6012
2,10,0.70,71.5,78.5,85.5,0.5993
11,50,0.50,73.5,82.5,85.5,0.5992


## 9. Table 4 (on a subset of queries)

In [12]:
import os
import re
import ast
import json
import time
import string
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import DPRReader, DPRReaderTokenizer
from pyserini.search import get_topics
from pyserini.search.lucene import LuceneSearcher

# --- Self-contained & CPU-optimized setup ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu':
    torch.set_num_threads(os.cpu_count() or 4)

RUNS_DIR = Path('runs')
INDEX_CACHE_DIR = Path(os.path.expanduser('~')) / '.cache' / 'pyserini' / 'indexes'
lucene_matches = sorted([p for p in INDEX_CACHE_DIR.glob('lucene-inverted.wikipedia-dpr-100w*') if p.is_dir()])
if not lucene_matches:
    raise FileNotFoundError(f'Local Lucene index not found in {INDEX_CACHE_DIR}')
root = str(lucene_matches[-1])

lucene_searcher = LuceneSearcher(root)
passage_cache = {}

ALL_DATASETS = {
    'nq':      {'topics': 'dpr-nq-test',      'tag': 'nq-test',      'regex': False},
    'trivia':  {'topics': 'dpr-trivia-test',  'tag': 'trivia-test',  'regex': False},
    'wq':      {'topics': 'dpr-wq-test',      'tag': 'wq-test',      'regex': False},
    'curated': {'topics': 'dpr-curated-test', 'tag': 'curated-test', 'regex': True},
    'squad':   {'topics': 'dpr-squad-test',   'tag': 'squad-test',   'regex': False},
}

PAPER_TABLE4_REF = {
    ('DPR-Single', 'NQ'): 41.5,
    ('DPR-Multi', 'NQ'): 41.5,
    ('DPR-Multi', 'TRIVIA'): 56.8,
    ('DPR-Multi', 'WQ'): 42.4,
    ('DPR-Multi', 'CURATED'): 49.4,
    ('DPR-Multi', 'SQUAD'): 24.1,
}

def load_trec_topk(trec_path: Path, max_k: int = 20):
    runs = {}
    with open(trec_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 6:
                qid, _, docid, _, score, _ = parts[:6]
                lst = runs.setdefault(qid, [])
                if len(lst) < max_k:
                    lst.append((docid, float(score)))
    return runs

def get_title_and_text(docid: str):
    if docid not in passage_cache:
        doc = lucene_searcher.doc(str(docid))
        if doc is None:
            passage_cache[docid] = ('', '')
        else:
            raw = json.loads(doc.raw())
            contents = raw.get('contents', '')
            parts = contents.split('\n', 1)
            if len(parts) == 2:
                passage_cache[docid] = (parts[0].strip().strip('"'), parts[1].strip())
            else:
                passage_cache[docid] = (raw.get('title', '').strip(), contents.strip())
    return passage_cache[docid]

def normalize_answer(s: str) -> str:
    s = ''.join(ch for ch in s.lower() if ch not in string.punctuation)
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    return ' '.join(s.split())

def check_exact_match(pred: str, raw_gold, use_regex: bool = False) -> bool:
    if isinstance(raw_gold, list):
        golds = [str(g) for g in raw_gold]
    elif isinstance(raw_gold, str) and raw_gold.startswith('['):
        try:
            golds = [str(g) for g in ast.literal_eval(raw_gold)]
        except Exception:
            golds = [raw_gold]
    else:
        golds = [str(raw_gold)]

    if use_regex:
        for pat in golds:
            try:
                if re.search(pat, pred, flags=re.IGNORECASE):
                    return True
            except re.error:
                if normalize_answer(pred) == normalize_answer(pat):
                    return True
        return False

    norm_p = normalize_answer(pred)
    return any(norm_p == normalize_answer(g) for g in golds)

@torch.no_grad()
def evaluate_table4_reader(
    trec_path: Path,
    topics_name: str,
    reader_model_name: str,
    use_regex: bool = False,
    top_k_passages: int = 10,
    batch_size: int = 10,
    max_seq_len: int = 220,
    max_span_len: int = 10,
    max_queries: int = 200,
):
    tokenizer = DPRReaderTokenizer.from_pretrained(reader_model_name)
    model = DPRReader.from_pretrained(reader_model_name).eval()
    if DEVICE == 'cpu':
        model = torch.quantization.quantize_dynamic(model, {torch.nn.Linear}, dtype=torch.qint8)
    else:
        model = model.to(DEVICE)

    topics = get_topics(topics_name)
    runs = load_trec_topk(trec_path, max_k=top_k_passages)
    qids = [q for q in topics if str(q) in runs]
    if max_queries is not None:
        qids = qids[:max_queries]

    em_hits = 0
    total_sec = 0.0

    for qid in tqdm(qids, desc=f'Table 4 ({topics_name} | {reader_model_name.split("/")[-1]})'):
        q_str = str(qid)
        question = topics[qid]['title']
        cands = [get_title_and_text(d) for d, _ in runs[q_str][:top_k_passages]]

        t0 = time.perf_counter()
        best_joint = -1e9
        best_ans = ''

        for i in range(0, len(cands), batch_size):
            batch = cands[i:i + batch_size]
            enc = tokenizer(
                questions=[question] * len(batch),
                titles=[t for t, _ in batch],
                texts=[p for _, p in batch],
                padding=True,
                truncation=True,
                max_length=max_seq_len,
                return_tensors='pt',
            ).to(DEVICE)

            out = model(**enc)
            rel_logits = out.relevance_logits.cpu().numpy()
            start_logits = out.start_logits.cpu().numpy()
            end_logits = out.end_logits.cpu().numpy()
            input_ids = enc['input_ids'].cpu().numpy()

            for bi in range(len(batch)):
                seps = np.where(input_ids[bi] == tokenizer.sep_token_id)[0]
                p_start = int(seps[1]) + 1 if len(seps) >= 2 else 1
                p_end = int(seps[2]) if len(seps) >= 3 else len(input_ids[bi])
                if p_start >= p_end:
                    continue

                s_log = start_logits[bi, p_start:p_end]
                e_log = end_logits[bi, p_start:p_end]
                L = len(s_log)
                span_mat = s_log[:, None] + e_log[None, :]
                mask = np.triu(np.ones((L, L), dtype=bool), 0) & np.tril(np.ones((L, L), dtype=bool), max_span_len - 1)
                span_mat = np.where(mask, span_mat, -1e9)

                si, ei = divmod(int(np.argmax(span_mat)), L)
                joint_score = float(rel_logits[bi]) + float(span_mat[si, ei])
                if joint_score > best_joint:
                    best_joint = joint_score
                    best_ans = tokenizer.decode(input_ids[bi, p_start + si : p_start + ei + 1], skip_special_tokens=True).strip()

        if DEVICE == 'cuda':
            torch.cuda.synchronize()
        total_sec += (time.perf_counter() - t0)
        em_hits += int(check_exact_match(best_ans, topics[qid]['answers'], use_regex=use_regex))

    del model
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

    em_pct = round(100.0 * em_hits / max(len(qids), 1), 2)
    lat_ms = round(1000.0 * total_sec / max(len(qids), 1), 2)
    return em_pct, lat_ms, len(qids)

# --- Fast Table 4 Evaluation with Incremental Checkpointing ---
READER_TOP_K = 10 if DEVICE == 'cpu' else 50
READER_BS = 10 if DEVICE == 'cpu' else 32
MAX_READER_QUERIES = 200 if DEVICE == 'cpu' else None

out_csv = RUNS_DIR / 'table4_reproduction.csv'
table4_rows = pd.read_csv(out_csv).to_dict('records') if out_csv.exists() else []
done_keys = {(r['Model'], r['Dataset']) for r in table4_rows}

tasks = [('DPR-Single', 'nq', RUNS_DIR / 'run.dpr.nq-test.dpr-single-nq.trec', 'facebook/dpr-reader-single-nq-base')]
for ds_key, ds_cfg in ALL_DATASETS.items():
    tasks.append(('DPR-Multi', ds_key, RUNS_DIR / f"run.dpr.{ds_cfg['tag']}.multi.trec", 'facebook/dpr-reader-multiset-base'))

for model_label, ds_key, trec_path, reader_ckpt in tasks:
    ds_upper = ds_key.upper()
    if (model_label, ds_upper) in done_keys:
        print(f'[Skip] Already computed {model_label} on {ds_upper}')
        continue
    if not trec_path.exists():
        print(f'[Skip] Missing {trec_path}')
        continue

    ds_cfg = ALL_DATASETS[ds_key]
    em, lat, n_q = evaluate_table4_reader(
        trec_path, ds_cfg['topics'], reader_ckpt,
        use_regex=ds_cfg['regex'], top_k_passages=READER_TOP_K,
        batch_size=READER_BS, max_queries=MAX_READER_QUERIES
    )
    table4_rows.append({
        'Model': model_label,
        'Dataset': ds_upper,
        'Queries': n_q,
        'Top-K Passages': READER_TOP_K,
        'Reproduced EM (%)': em,
        'Paper Table 4 EM (%)': PAPER_TABLE4_REF.get((model_label, ds_upper)),
        'Reader Latency (ms/q)': lat,
    })
    # Save immediately after each dataset
    table4_df = pd.DataFrame(table4_rows)
    table4_df.to_csv(out_csv, index=False)
    print(f'[Saved] {model_label} | {ds_upper}: EM = {em}% (Paper: {PAPER_TABLE4_REF.get((model_label, ds_upper))}%)')

table4_df = pd.DataFrame(table4_rows)
display(table4_df)
display(table4_df.pivot(index='Model', columns='Dataset', values='Reproduced EM (%)'))

[Skip] Already computed DPR-Single on NQ
[Skip] Already computed DPR-Multi on NQ


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] DPRReader LOAD REPORT from: facebook/dpr-reader-multiset-base
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
span_predictor.encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 
span_predictor.encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\Dwija Patel\AppData\Local\Temp\ipykernel_29488\2959570874.py:119: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e q

Table 4 (dpr-trivia-test | dpr-reader-multiset-base):   0%|          | 0/200 [00:00<?, ?it/s]

c:\Users\Dwija Patel\dwija\sem5\indexing\winenv\Lib\site-packages\transformers\tokenization_utils_base.py:2355: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


[Saved] DPR-Multi | TRIVIA: EM = 45.5% (Paper: 56.8%)


... (14 outputs left out)

c:\Users\Dwija Patel\dwija\sem5\indexing\winenv\Lib\site-packages\transformers\tokenization_utils_base.py:2355: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


[Saved] DPR-Multi | SQUAD: EM = 6.0% (Paper: 24.1%)


,Model,Dataset,Queries,Reproduced EM (%),Paper Table 4 EM (%),Reader Latency (ms/q),Top-K Passages
0,DPR-Single,NQ,500,28.6,41.5,7735.24,NaN
1,DPR-Multi,NQ,500,12.0,41.5,7479.89,NaN
2,DPR-Multi,TRIVIA,200,45.5,56.8,2123.47,10.0
3,DPR-Multi,WQ,200,13.5,42.4,1706.95,10.0
4,DPR-Multi,CURATED,200,41.5,49.4,1698.20,10.0
5,DPR-Multi,SQUAD,200,6.0,24.1,1737.23,10.0


Dataset,CURATED,NQ,SQUAD,TRIVIA,WQ
Model,,,,,
DPR-Multi,41.5,12.0,6.0,45.5,13.5
DPR-Single,NaN,28.6,NaN,NaN,NaN
